In [9]:
import cv2
import numpy as np
from ultralytics import YOLO
import contextlib
import io

# ------------------------------
# SETTINGS
# ------------------------------
VIDEO_PATH = r"C:\Users\patel\Downloads\straight_lane (1).mp4"
OUTPUT_PATH = "output_detection_distance.mp4"
MODEL_PATH = "yolov8n.pt"
model = YOLO(MODEL_PATH)
model.to(device)
model.overrides['verbose'] = False

CONF_THRESHOLD = 0.4
DISTANCE_CONSTANT = 8000  # Adjust for distance tuning
device = "cpu"

model = YOLO(MODEL_PATH, verbose=False).to(device)  # Already set, but added redirect for extra suppression

# Initialize video
cap = cv2.VideoCapture(VIDEO_PATH)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

# Simple tracking for ID consistency
track_memory = {}  # id -> last_center
id_counter = 0
FONT = cv2.FONT_HERSHEY_SIMPLEX

def estimate_distance(h):
    return DISTANCE_CONSTANT / max(h, 1)

# ------------------------------
# MAIN LOOP
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Suppress YOLO console output
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        results = model(frame, conf=CONF_THRESHOLD, verbose=False)

    new_tracks = []
    
    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            if model.names[cls] not in ["car", "truck", "bus", "motorbike"]:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            h = y2 - y1
            distance = estimate_distance(h)

            # Simple ID matching by proximity
            matched_id = None
            for tid, last_center in track_memory.items():
                dist = np.linalg.norm(np.array([cx, cy]) - np.array(last_center))
                if dist < 50:  # Proximity threshold
                    matched_id = tid
                    break

            if matched_id is None:
                matched_id = id_counter
                id_counter += 1

            track_memory[matched_id] = (cx, cy)  # Update last position
            new_tracks.append(matched_id)

            # Visualization: Rectangle, ID, class, and distance
            color = (0, 255, 0)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID {matched_id}", (x1, y1 - 25), FONT, 0.6, (255, 255, 0), 2)
            cv2.putText(frame, f"{model.names[cls]} {distance:.1f}m", (x1, y1 - 5), FONT, 0.6, (0, 255, 0), 2)

    # Cleanup old tracks (simple: remove if not seen)
    for tid in list(track_memory.keys()):
        if tid not in new_tracks:
            del track_memory[tid]

    cv2.imshow("Detection + Distance", frame)
    out.write(frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()
print(f"✅ Clean video saved as {OUTPUT_PATH}")


✅ Clean video saved as output_detection_distance.mp4
